# 1) Importação de Bibliotecas

In [11]:
import json
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import randint

from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, cross_val_score, cross_val_predict, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)


pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [12]:
root = Path.cwd()
meta_data_dir = root.parent
data_dir = root.parent / "model" / "data"

# 2) SetUp de Dados

In [13]:
train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")
metadata = json.loads((meta_data_dir / "model" / "metadata.json").read_text())

target_column = metadata['target']

x_train = train.drop(columns=[target_column], axis=1)
y_train = train[target_column]
x_test = test.drop(columns=[target_column], axis=1)
y_test = test[target_column]

class_order = metadata["classes"]
print(f"Train: {x_train.shape}  |  Test: {x_test.shape}")
print(f"Classes ({len(class_order)}): {class_order}")
print(f"Distribuição no train:\n{y_train.value_counts()}")

Train: (14609, 55)  |  Test: (3653, 54)
Classes (12): ['ARP_poisioning', 'DDOS_Slowloris', 'DOS_SYN_Hping', 'MQTT_Publish', 'Metasploit_Brute_Force_SSH', 'NMAP_FIN_SCAN', 'NMAP_OS_DETECTION', 'NMAP_TCP_scan', 'NMAP_UDP_SCAN', 'NMAP_XMAS_TREE_SCAN', 'Thing_Speak', 'Wipro_bulb']
Distribuição no train:
Attack_type
Thing_Speak                   6073
ARP_poisioning                4232
MQTT_Publish                  3313
DDOS_Slowloris                 425
Wipro_bulb                     175
NMAP_UDP_SCAN                  154
NMAP_XMAS_TREE_SCAN             64
NMAP_OS_DETECTION               60
DOS_SYN_Hping                   46
NMAP_TCP_scan                   35
Metasploit_Brute_Force_SSH      28
NMAP_FIN_SCAN                    4
Name: count, dtype: int64


# 3) Treinamento dos modelos

In [ ]:
# Definição das camadas de teste
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
# Modelo de Aprendizagem de Máquina
rf_model = RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample", random_state=42, n_jobs=-1,)

In [ ]:
scores = cross_validate(rf_model, X=x_train, y=y_train, cv=cv, 
                       scoring=["accuracy", "balanced_accuracy", "f1_macro", "f1_weighted"])

c:\Users\rpf97\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
params = {
    "model__n_estimators": randint(200, 800),
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": randint(1, 10),
    "class_weight": ["balanced", "balanced_subsample", None],
    "max_features": ["sqrt", "log2", None]
}

search = RandomizedSearchCV(rf_model, params, n_iter=20, cv=cv, scoring="f1_macro", n_jobs=-1, random_state=42, verbose=1).fit(x_train, y_train)

best_option = search.best_estimator_

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\rpf97\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
c:\Users\rpf97\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [19]:
y_test_pred = best_option.predict(x_test)

print(f'Classification Report: {classification_report(y_test, y_test_pred, zero_division=0)}')
print('Macro F1: ', f1_score(y_test, y_test_pred, average="macro"))
print('Balanced Accuracy Score: ', balanced_accuracy_score(y_test, y_test_pred))

cm = confusion_matrix(y_test, y_test_pred, normalize="true")
ConfusionMatrixDisplay(cm, display_labels=best_option.classes_).plot()
plt.show()

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- service_radius
